# 03 · Zone and hub panels
We build the two datasets of the analysis, with all prices in real 2025 dollars:
- **zone panel**: 20 PJM zones × 91 months (January 2019 – July 2026) = 1,820 rows, with energy and capacity prices, load, data center exposure and controls;
- **hub panel**: 5 hubs × 91 months = 455 rows, for the system-wide comparison (Track 2).

Outputs: `panel_zone_month.parquet`, `panel_hub_month.parquet` (Section 2.1.2 of the thesis).

July 2026 is the last month where every control exists.

In [1]:
import glob
import numpy as np
import pandas as pd
from utils import RAW, CLEAN, ZONES, ZONE_STATE, EMAAC, MAAC, HUBS, RGGI_STATES, STATE_NAME, NOAA_CODE

START, END = "2019-01-01", "2026-07-01"
months = pd.date_range(START, END, freq="MS")

## Day-ahead prices by zone
From hourly PJM prices to monthly averages: total LMP, its energy and congestion components, and the on-peak average (weekdays, hours starting 7:00 to 22:00). Only current rows are kept and each hour is counted once. We also check that every zone-month has all its hours.

In [2]:
cols = ["datetime_beginning_utc", "datetime_beginning_ept", "pnode_name", "total_lmp_da",
        "system_energy_price_da", "congestion_price_da", "row_is_current"]
lmp = pd.concat([pd.read_csv(f, usecols=cols) for f in sorted(glob.glob(str(RAW / "pjm_lmp/*.csv.gz")))])
lmp = lmp[lmp.row_is_current].copy()
key = ["datetime_beginning_utc", "pnode_name"]
assert lmp.groupby(key).total_lmp_da.nunique().le(1).all(), "Conflicting current price revisions"
lmp = lmp.drop_duplicates(key)
lmp = lmp[lmp.pnode_name.isin(ZONES)].rename(columns={"pnode_name": "zone"})

t = pd.to_datetime(lmp.datetime_beginning_ept, format="%m/%d/%Y %I:%M:%S %p")
lmp["month"] = t.dt.to_period("M").dt.to_timestamp()
lmp["onpeak"] = (t.dt.weekday < 5) & t.dt.hour.between(7, 22)

prices = lmp.groupby(["zone", "month"]).agg(lmp=("total_lmp_da", "mean"),
                                            energy=("system_energy_price_da", "mean"),
                                            congestion=("congestion_price_da", "mean"),
                                            hours=("total_lmp_da", "size")).reset_index()
onpeak = lmp[lmp.onpeak].groupby(["zone", "month"]).total_lmp_da.mean().rename("lmp_onpeak")
prices = prices.join(onpeak, on=["zone", "month"])
print(prices[prices.month <= END].hours.min(), "hours in the thinnest zone-month (should be about 672-744)")

# hours in each month in Eastern time (the daylight-saving changes give 743 or 745 hours)
start = prices.month.dt.tz_localize("America/New_York")
end = (prices.month + pd.offsets.MonthBegin()).dt.tz_localize("America/New_York")
prices["expected_hours"] = ((end - start) / pd.Timedelta(hours=1)).astype(int)
prices["valid_price_hours"] = lmp.groupby(["zone", "month"]).total_lmp_da.count().values
prices["price_complete"] = prices.valid_price_hours.eq(prices.expected_hours)

672 hours in the thinnest zone-month (should be about 672-744)


## Hub prices
Monthly average of the daily on-peak price at the five hubs of Track 2.

In [3]:
ice = pd.concat([pd.read_excel(f) for f in sorted(glob.glob(str(RAW / "hub_prices/*.xlsx")))])
name = {v["ice"]: k for k, v in HUBS.items()}
ice = ice[ice["Price hub"].isin(name)]
ice = ice.drop_duplicates().copy()
ice["hub"] = ice["Price hub"].map(name)
ice["month"] = pd.to_datetime(ice["Delivery start date"]).dt.to_period("M").dt.to_timestamp()
hub_prices = ice.groupby(["hub", "month"])["Wtd avg price $/MWh"].mean().rename("price").reset_index()
print(hub_prices.groupby("hub").month.count().to_string())

hub
Mass Hub      93
Mid-C         93
PJM West      93
Palo Verde    93
SP15          93


## Capacity prices
Each zone pays the price of its own capacity area if it cleared separately, otherwise EMAAC, MAAC or the RTO price. Delivery years run from June to May.

In [4]:
cap = pd.read_csv(RAW / "manual/pjm_capacity_lda.csv")
cap_lookup = cap.set_index(["delivery_year", "lda"]).rcp_usd_mw_day

def capacity_price(zone, delivery_year):
    for area in [zone, "EMAAC" if zone in EMAAC else None, "MAAC" if zone in MAAC else None, "RTO"]:
        if (delivery_year, area) in cap_lookup:
            return cap_lookup[(delivery_year, area)]

def delivery_year(m):
    y = m.year if m.month >= 6 else m.year - 1
    return f"{y}/{y + 1}"

cap_zone = pd.DataFrame([(z, m, capacity_price(z, delivery_year(m))) for z in ZONES for m in months],
                        columns=["zone", "month", "capacity_usd_mw_day"])
assert cap_lookup.index.is_unique, 'Duplicate capacity price keys'
assert cap_zone.capacity_usd_mw_day.notna().all(), 'Missing capacity price'

## Load
Hourly EIA-930 load by zone, without the few bad readings (outside 20%–300% of the zone median). The 2021 peak is the denominator of the exposure, and the 2021 peak-to-average ratio converts capacity prices to $/MWh.

In [5]:
SUBBA = {"AE": "AECO", "AP": "APS", "BC": "BGE", "CE": "COMED", "JC": "JCPL", "ME": "METED", "PE": "PECO",
         "PEP": "PEPCO", "PL": "PPL", "PN": "PENELEC", "PS": "PSEG"}
load = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(str(RAW / "load/*.parquet")))])
assert not load.duplicated(["subba", "period"]).any(), "Duplicate load hours"
load["zone"] = load.subba.replace(SUBBA)
load["value"] = pd.to_numeric(load.value, errors="coerce")
med = load.groupby("zone").value.transform("median")
load["retained"] = (load.value > 0.2 * med) & (load.value < 3 * med)
load.loc[~load.retained].to_csv(CLEAN / "load_rejected.csv", index=False)
load = load[load.retained].copy()
load["month"] = pd.to_datetime(load.period.str[:7])
load_m = load.groupby(["zone", "month"]).value.agg(load_mw="mean", peak_mw="max", valid_load_hours="count").reset_index()
load_m["load_complete"] = load_m.valid_load_hours.eq(load_m.month.dt.days_in_month * 24)

y2021 = load_m[load_m.month.dt.year == 2021].groupby("zone")
base = pd.DataFrame({"peak_2021_mw": y2021.peak_mw.max(),
                     "peak_to_avg": y2021.peak_mw.max() / load[load.month.dt.year == 2021].groupby("zone").value.mean()})
print(base.round(2).sort_values("peak_2021_mw", ascending=False).head(8).to_string())
base["baseline_load_complete"] = y2021.load_complete.all()

       peak_2021_mw  peak_to_avg
zone                            
AEP           22021         1.51
COMED         21204         1.95
DOM           20493         1.65
ATSI          12658         1.68
PSEG          10064         2.06
APS            9016         1.59
PECO           8424         1.90
PPL            7666         1.63


## Gas price
Gas price paid by power plants in each state. Missing months are interpolated; a state with gaps uses the US series.

In [6]:
gas = pd.read_excel(RAW / "gas/gas_price_power_plants_by_state.xls", sheet_name="Data 1", header=2)
gas["month"] = pd.to_datetime(gas.Date).dt.to_period("M").dt.to_timestamp()
gas = gas.set_index("month").drop(columns="Date")
gas.columns = [c.split(" Natural Gas")[0] for c in gas.columns]
gas_raw = gas.reindex(months).copy()
gas = gas_raw.interpolate(limit_direction="both")

def gas_price(state):
    s = gas.get(STATE_NAME[state])
    return s if s is not None and s.notna().all() else gas["U.S."]

## Weather
Heating and cooling degree days by state (NOAA).

In [7]:
w = pd.read_parquet(RAW / "weather/noaa_degree_days_state.parquet")
w["month"] = pd.to_datetime(dict(year=w.year, month=w.month, day=1))
weather = {s: w[w.state_code == NOAA_CODE[s]].set_index("month")[["hdd", "cdd"]] for s in STATE_NAME}
assert not w.duplicated(['state_code','month']).any(), 'Duplicate weather months'

## Renewables
Wind + solar share of generation for a group of states.

In [8]:
xl = pd.ExcelFile(RAW / "generation/generation_monthly.xlsx")
gen = pd.concat([pd.read_excel(xl, s, header=4) for s in xl.sheet_names if s[:4].isdigit() and int(s[:4]) >= 2019])
gen = gen[gen["TYPE OF PRODUCER"] == "Total Electric Power Industry"]
gen["month"] = pd.to_datetime(dict(year=gen.YEAR, month=gen.MONTH, day=1))
gen = gen.pivot_table(index=["STATE", "month"], columns="ENERGY SOURCE", values="GENERATION\n(Megawatthours)", aggfunc="sum")

def renewable_share(states):
    g = gen[gen.index.get_level_values("STATE").isin(states)].groupby("month").sum()
    return (g["Wind"] + g["Solar Thermal and Photovoltaic"]) / g["Total"]

## RGGI carbon price
Auction price carried forward between auctions, for member states only. Virginia was a member from 2021 to 2023 and again from July 2026; New Jersey from 2020.

In [9]:
rggi = pd.read_csv(RAW / "manual/rggi_auctions.csv", parse_dates=["date"])
rggi = rggi.set_index(rggi.date.dt.to_period("M").dt.to_timestamp()).clearing_price_usd
rggi = rggi.reindex(pd.date_range("2018-01-01", END, freq="MS")).ffill().reindex(months)

def rggi_price(state):
    member = pd.Series(state in RGGI_STATES, index=months)
    if state == "VA":
        member = ((months >= "2021-01-01") & (months <= "2023-12-01")) | (months >= "2026-07-01")
    if state == "NJ":
        member = months >= "2020-01-01"
    return rggi * member

## Controls
One table of controls for a state (gas, weather, carbon) and a region (generation mix).

In [10]:
def controls(state, region):
    state_name = STATE_NAME[state]
    fallback = state_name not in gas or not gas[state_name].notna().all()
    return pd.DataFrame({'gas': gas_price(state), 'hdd': weather[state].hdd,
        'cdd': weather[state].cdd, 'renew_share': renewable_share(region),
        'rggi': rggi_price(state),
        'gas_imputed': gas_raw.get(state_name, pd.Series(np.nan,index=months)).isna(),
        'gas_national_fallback': fallback}, index=months)

## Data center exposure
Operating sites enter the zone's capacity in January of their operating year. Sites without a year are treated as already operating before 2019 (2018), so they do not add to the growth. We also attach PJM's large-load adjustments (from 2022), a second measure that does not depend on our list.

In [11]:
inventory = pd.read_csv(CLEAN / "dc_unified.csv")
assert inventory.dc_id.is_unique, "Duplicate site IDs"
assert inventory.pjm_zone.dropna().isin(ZONES).all(), "Unknown zone label"
inventory["year_numeric"] = pd.to_numeric(inventory.operating_year, errors="coerce").fillna(2018)
inventory["year_valid"] = inventory.year_numeric.between(1900, 2026) & inventory.year_numeric.mod(1).eq(0)
inventory["capacity_valid"] = np.isfinite(inventory.facility_mw) & inventory.facility_mw.gt(0)
inventory["estimated"] = inventory.mw_source.str.contains("estimated", case=False, na=False)
inventory["analysis_eligible"] = (inventory.status.eq("operating") & inventory.pjm_zone.notna()
                                  & inventory.year_valid & inventory.capacity_valid)
inventory["exclusion_reason"] = np.select(
    [inventory.status.ne("operating"), inventory.pjm_zone.isna(), ~inventory.year_valid, ~inventory.capacity_valid],
    ["nonoperating", "unassigned_zone", "unknown_or_invalid_year", "invalid_capacity"], default="included")
inventory.to_csv(CLEAN / "dc_record_ledger.csv", index=False)

dc = inventory[inventory.analysis_eligible].copy()
dc["start"] = pd.to_datetime(dc.year_numeric.astype(int).astype(str) + "-01-01")
dc_mw = pd.DataFrame([(z, m, dc.loc[dc.pjm_zone.eq(z) & dc.start.le(m), "facility_mw"].sum()) for z in ZONES for m in months],
                     columns=["zone", "month", "dc_mw"])
dc_mw["dc_unknown_timing_mw"] = 0.0     # no operating site is left without a start year
dc_mw["dc_timing_assumption"] = "january"
dc_mw["dc_is_inventory_subtotal"] = True

# PJM large-load adjustments: for each target year, the report published that year (or the latest one)
ll = pd.read_csv(RAW / "manual/pjm_large_load_adjustments.csv")
ll = ll[ll.vintage == ll.target_year.clip(upper=ll.vintage.max())].copy()
assert not ll.duplicated(["zone_name", "target_year"]).any()
ll_year = ll.set_index(["zone_name", "target_year"])
for field, output in [("adj_mw", "pjm_large_load_mw"), ("vintage", "pjm_large_load_vintage"),
                      ("above_embedded", "pjm_large_load_above_embedded")]:
    dc_mw[output] = [ll_year[field].get((z, m.year), np.nan) for z, m in zip(dc_mw.zone, dc_mw.month)]
dc_mw["pjm_large_load_observed"] = dc_mw.pjm_large_load_mw.notna()

## Zone panel
All pieces are joined on the full zone × month grid. Capacity in $/MWh = $/MW-day ÷ 24 × 2021 peak-to-average ratio; all-in price = LMP + capacity; exposure = data center MW ÷ 2021 peak load. Prices are converted to 2025 dollars with the CPI.

In [12]:
cpi = pd.read_csv(RAW / "macro/cpi.csv", parse_dates=["observation_date"]).set_index("observation_date").CPIAUCSL
cpi_raw = cpi.copy()
cpi = cpi.reindex(pd.date_range("2018-01-01", END, freq="MS")).interpolate()   # Missing CPI is interpolated; provenance retained below.
to_2025 = (cpi[cpi.index.year == 2025].mean() / cpi).reindex(months).rename("to_2025")

grid = pd.MultiIndex.from_product([ZONES, months], names=['zone','month']).to_frame(index=False)
zone = (grid.merge(prices, on=['zone','month'], how='left', validate='one_to_one')
        .merge(cap_zone, on=['zone','month'], how='left', validate='one_to_one').merge(load_m, on=['zone','month'], how='left', validate='one_to_one')
        .merge(dc_mw, on=["zone", "month"], how="left", validate="one_to_one").join(base, on="zone").join(to_2025, on="month"))
zone = zone.merge(pd.concat([controls(ZONE_STATE[z], [ZONE_STATE[z]]).assign(zone=z) for z in ZONES])
                  .rename_axis("month").reset_index(), on=["zone", "month"], how="left", validate="one_to_one")

zone["capacity_mwh"] = zone.capacity_usd_mw_day / 24 * zone.peak_to_avg
for c in ["lmp", "lmp_onpeak", "energy", "congestion", "capacity_mwh", "gas", "rggi"]:
    zone[c + "_real"] = zone[c] * zone.to_2025
zone["wholesale_real"] = zone.lmp_real + zone.capacity_mwh_real
zone["dc_share"] = zone.dc_mw / zone.peak_2021_mw
zone["state"] = zone.zone.map(ZONE_STATE)
zone = zone[(zone.month >= START) & (zone.month <= END)].sort_values(["zone", "month"]).reset_index(drop=True)

print(zone.shape, "| missing values per column:", zone.isna().sum()[lambda x: x > 0].to_dict())
zone['cpi_imputed'] = zone.month.map(cpi_raw.reindex(months).isna())
zone['controls_complete'] = zone[['gas_real','hdd','cdd','renew_share','rggi_real']].notna().all(axis=1)
zone['dc_event_date_exact'] = False
assert len(zone) == len(ZONES)*len(months) and not zone.duplicated(['zone','month']).any()
assert zone[['lmp_real','capacity_mwh_real','peak_2021_mw']].notna().all().all()
zone.to_parquet(CLEAN / "panel_zone_month.parquet", index=False)

(1820, 45) | missing values per column: {'pjm_large_load_mw': 720, 'pjm_large_load_vintage': 720, 'pjm_large_load_above_embedded': 720}


## Hub panel
Same controls at the hub's state and region; PJM West is the treated hub.

In [13]:
hub = pd.concat([controls(v["state"], v["region"]).assign(hub=h) for h, v in HUBS.items()]).rename_axis("month").reset_index()
hub = hub.merge(hub_prices, on=["hub", "month"], how="left", validate="one_to_one").join(to_2025, on="month")
for c in ["price", "gas", "rggi"]:
    hub[c + "_real"] = hub[c] * hub.to_2025
hub["treated"] = (hub.hub == "PJM West").astype(int)

print(hub.shape, "| months without a price:", hub.price.isna().groupby(hub.hub).sum().to_dict())
hub['controls_complete'] = hub[['gas_real','hdd','cdd','renew_share','rggi_real']].notna().all(axis=1)
hub['product'] = hub.hub.map({h:v['ice'] for h,v in HUBS.items()})
hub['comparison_group_validated'] = False
assert len(hub)==len(HUBS)*len(months) and not hub.duplicated(['hub','month']).any()
assert hub.price.notna().all()
hub.to_parquet(CLEAN / "panel_hub_month.parquet", index=False)

(455, 15) | months without a price: {'Mass Hub': 0, 'Mid-C': 0, 'PJM West': 0, 'Palo Verde': 0, 'SP15': 0}


## How much prices increased
Change between the 2019–21 average and the latest 12 months (August 2025 – July 2026): +59.5 $/MWh in Dominion (Table 2.3 of the thesis).

In [14]:
def period(df):
    return np.where(df.month <= "2021-12-01", "2019-21", np.where(df.month >= "2025-08-01", "latest 12m", None))

z = zone.assign(period=period(zone)).dropna(subset=["period"])
change = z.pivot_table(index="zone", columns="period", values=["lmp_real", "capacity_mwh_real", "wholesale_real"])
change = change.swaplevel(axis=1)["latest 12m"] - change.swaplevel(axis=1)["2019-21"]
change["dc_share_now"] = zone[zone.month == END].set_index("zone").dc_share
print(change.sort_values("dc_share_now", ascending=False, kind="stable").round(1).to_string())

         capacity_mwh_real  lmp_real  wholesale_real  dc_share_now
zone                                                              
DOM                   19.5      40.0            59.5           0.8
AEP                    8.9      18.0            26.9           0.2
PSEG                   5.3      19.9            25.2           0.1
AECO                   6.6      19.4            26.0           0.1
COMED                  2.5       8.8            11.3           0.1
JCPL                   7.0      19.5            26.5           0.0
ATSI                   9.3      16.6            25.9           0.0
DEOK                   8.7      18.5            27.2           0.0
DUQ                   10.7      15.2            25.9           0.0
BGE                   21.9      40.7            62.6           0.0
PEPCO                 10.4      39.8            50.2           0.0
PECO                   5.5      19.0            24.5           0.0
PPL                    9.3      16.3            25.5          